# Explore SPECFEM2D model outputs — cleaned workflow

This notebook is the cleaned companion to `specfem_output_tools_refactored.py` and `seismic_gather_utils_refactored.py`.

It intentionally keeps the notebook focused on **configuration, execution, and QC figures**. Reusable functions have been moved out of the notebook and into Python modules with docstrings.

The workflow can:

1. discover SPECFEM2D `OUTPUT_FILES` directories;
2. load each gather, preferring binary SU files where available and falling back to SPECFEM ASCII files;
3. convert model gathers to standardized ObsPy `Stream` objects;
4. write intermediate SEG-Y files for downstream QC and comparison;
5. generate wiggle plots;
6. difference model gathers; and
7. run optional f-k, AGC, NMO/diffraction-scan, and source-wavelet experiments.

Generated products should be written outside the code repository.


## 1. Configuration

Edit these paths for your machine. `repo_dir` should be the parent directory containing the local project packages (`segy_tools/`, `specfem_tools/`, and the two refactored helper modules if they are not installed with `pip install -e .`).


In [ ]:
from pathlib import Path
import sys

# Parent directory containing segy_tools/, specfem_tools/, and local helper modules.
repo_dir = Path("~/Developer/karstgeo").expanduser()

# Root directory containing SPECFEM2D model outputs.
specfem_root = Path(
    "~/Library/CloudStorage/Box-Box/thompsong/"
    "2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d"
).expanduser()

# Choose frequency/model folder.
freq = "300Hz"  # alternatives used in the project: "10Hz", "50Hz", "300Hz"
model_root = specfem_root / "felix" / "highres_mesh" / freq

# Notebook-generated products. Keep these outside the code repository.
outdir = specfem_root / "glenn2" / freq
fig_dir = outdir / "figures" / "explore_specfem_output"
segy_out_dir = outdir / "segy" / "specfem_models"
diff_fig_dir = outdir / "figures" / "specfem_model_differences"

default_receiver_spacing_m = 1.0
default_first_receiver_x_m = 1.0
default_source_x_m = 0.0

default_tmin = 0.0
default_tmax = 0.3
wiggle_scale = 0.02
normalize_wiggles = False

for d in [fig_dir, segy_out_dir, diff_fig_dir]:
    d.mkdir(parents=True, exist_ok=True)

print(f"repo_dir     = {repo_dir}")
print(f"model_root   = {model_root}")
print(f"outdir       = {outdir}")
print(f"fig_dir      = {fig_dir}")
print(f"segy_out_dir = {segy_out_dir}")

## 2. Imports

The notebook avoids `os.chdir()`. If the local packages have not been installed, `repo_dir` is added to `sys.path`.


In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
sys.path.insert(0, str(repo_dir / "lib"))

print(sys.path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import obspy
from obspy import Stream

from specfem_output_tools_refactored import (
    SpecfemExportConfig,
    batch_write_model_products,
    find_specfem_model_outputs,
    model_number_from_name,
    model_name_from_output_dir,
    plot_model_difference_from_segy,
    plot_segy_file,
    plot_su_directory,
    write_model_products,
)

from seismic_gather_utils_refactored import (
    apply_agc,
    apply_fk_velocity_filter,
    apply_linear_time_gain,
    gaussian_taper,
    normalize,
    plot_fk_spectrum,
    plot_nmo_velocity_grid,
    ricker_wavelet,
)

# Existing project plotting function, if available.
from segy_tools.gather import plot_wiggle_gather_from_stream

config = SpecfemExportConfig(
    segy_out_dir=segy_out_dir,
    fig_dir=fig_dir,
    diff_fig_dir=diff_fig_dir,
    receiver_spacing_m=default_receiver_spacing_m,
    first_receiver_x_m=default_first_receiver_x_m,
    source_x_m=default_source_x_m,
    network="SY",
)
config.ensure_directories()

print("Imported cleaned helper modules and created export config.")

## 3. Discover model output directories

The default discovery pattern matches the single-letter model folders used in several of the karst SPECFEM2D experiments: `A/OUTPUT_FILES`, `B/OUTPUT_FILES`, etc. Change the pattern if using `Mod10/OUTPUT_FILES` style names.


In [ ]:
model_output_dirs = find_specfem_model_outputs(model_root, pattern="[A-Z]/OUTPUT_FILES")

df_models = pd.DataFrame({
    "model": [p.parent.name for p in model_output_dirs],
    "model_number": [model_number_from_name(p.parent.name) for p in model_output_dirs],
    "output_dir": model_output_dirs,
}).sort_values(["model_number", "model"], na_position="last").reset_index(drop=True)

display(df_models)

## 4. Convert and plot model gathers

This batch processor loads each SPECFEM2D `OUTPUT_FILES` directory, preferring SU files where available. It then writes standardized SEG-Y and QC wiggle plots.


In [ ]:
processed_df = batch_write_model_products(
    model_output_dirs=model_output_dirs,
    config=config,
    components=("BXZ", "BXX"),
    extension="semv",
    prefer_su=True,
    write_segy_file=True,
    make_plot=True,
    tmin=default_tmin,
    tmax=default_tmax,
    scale=wiggle_scale,
    normalize=normalize_wiggles,
    verbose=True,
)

display(processed_df)

## 5. Plot individual SEG-Y or SU files

Use these cells for single-file debugging before or after the batch conversion.


In [ ]:
# Example: plot one converted SEG-Y file.
example_segy = segy_out_dir / "A_BXZ.segy"
if example_segy.exists():
    st = plot_segy_file(
        example_segy,
        config=config,
        tmin=default_tmin,
        tmax=default_tmax,
        scale=wiggle_scale,
        normalize=normalize_wiggles,
    )
else:
    print(f"Example SEG-Y not found: {example_segy}")

In [ ]:
# Optional: plot raw SU files from a directory.
# su_df = plot_su_directory(Path("~/Downloads").expanduser(), outfile_dir=fig_dir / "su_wiggles")
# display(su_df)

## 6. Difference two model gathers

This reads two converted SEG-Y files, trims to common trace/sample dimensions, computes `A - B`, and plots the inputs plus the difference. Difference SEG-Y export is optional.


In [ ]:
for model_a in ["A", "B", "C", "D"]:
    try:
        fig, diff = plot_model_difference_from_segy(
            model_a=model_a,
            model_b="E",
            component="BXZ",
            config=config,
            diff_segy_dir=segy_out_dir / "differences",
            tmin=0.0,
            tmax=0.3,
        )
        print(f"{model_a} - E: diff matrix shape = {diff.shape}")
    except FileNotFoundError as exc:
        print(f"Skipping {model_a} - E: {exc}")

## 7. Optional gather conditioning: f-k, AGC, and NMO/diffraction scan

These functions are **not SPECfEM-specific**. They operate on ObsPy gathers and are good candidates for later migration into `segy_tools`.

Use f-k filtering cautiously when hunting diffractions: the filter can remove diffraction wings if they share the muted apparent-velocity range.


In [ ]:
# Load an example gather if it was not already loaded above.
if "st" not in globals() and example_segy.exists():
    st = obspy.read(str(example_segy), format="SEGY", unpack_trace_headers=True)

if "st" in globals():
    plot_fk_spectrum(st, receiver_spacing_m=config.receiver_spacing_m, max_display_freq_hz=600.0, title="Raw gather f-k spectrum")

    fk_st = apply_fk_velocity_filter(
        st,
        min_velocity_mps=1000.0,
        receiver_spacing_m=config.receiver_spacing_m,
        use_taper=True,
        taper_width_mps=200.0,
    )
    plot_fk_spectrum(fk_st, receiver_spacing_m=config.receiver_spacing_m, max_display_freq_hz=600.0, title="Filtered gather f-k spectrum")
    plot_wiggle_gather_from_stream(fk_st, normalize=True, scale=1.0, title="f-k filtered gather")
else:
    print("No example stream loaded.")

In [ ]:
if "st" in globals():
    agc_st = apply_agc(st, window_sec=0.02)
    tgain_st = apply_linear_time_gain(st, power=1.0)

    trial_velocities = [340, 1000, 1300, 1600, 2200, 3000, 10000]
    fig = plot_nmo_velocity_grid(
        st=agc_st,
        trial_velocities_mps=trial_velocities,
        source_x_m=150.0,
        receiver_spacing_m=1.0,
        first_receiver_x_m=1.0,
        offset_range_m=(-50.0, 50.0),
        clip_percentile=95.0,
        cols_per_row=3,
    )

## 8. Source wavelet experiment

This diagnostic cell compares Ricker wavelets and filtered/tapered versions. It is useful for thinking about source-time functions but is not specific to SPECFEM2D output files.


In [ ]:
from obspy.signal.filter import bandpass

dt = 0.0001
duration = 0.6

t, ricker_300 = ricker_wavelet(f0_hz=300.0, dt_s=dt, duration_s=duration)
_, ricker_10 = ricker_wavelet(f0_hz=10.0, dt_s=dt, duration_s=duration)

ricker_300_bp_5_100 = normalize(bandpass(ricker_300, freqmin=5.0, freqmax=100.0, df=1.0 / dt, corners=4, zerophase=True))
ricker_300_bp_8_15 = normalize(bandpass(ricker_300, freqmin=8.0, freqmax=15.0, df=1.0 / dt, corners=4, zerophase=True))
ricker_300_bp_8_15_tapered = normalize(ricker_300_bp_8_15 * gaussian_taper(t, sigma_s=0.04))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t, ricker_10, label="10 Hz Ricker", linewidth=2)
ax.plot(t, ricker_300, label="300 Hz Ricker", linewidth=1.5)
ax.plot(t, ricker_300_bp_5_100, label="300 Hz Ricker bandpassed 5–100 Hz", linewidth=1.5)
ax.plot(t, ricker_300_bp_8_15_tapered, label="300 Hz Ricker bandpassed 8–15 Hz + taper", linewidth=2)
ax.set_xlim(-0.15, 0.15)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Normalized amplitude")
ax.set_title("Ricker wavelet filtering/tapering experiment")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

## 9. Practical notes

- Keep exact field SEG-2/SEG-D/SEG-Y/SU format tools in `segy_tools`, not in SPECfEM-specific modules.
- Keep SPECFEM ASCII readers and SPECFEM model discovery/export wrappers in SPECfEM modules.
- Treat f-k filtering, AGC, NMO scans, wavelets, gather differencing, and wiggle plotting as general seismic gather utilities.
- For diffraction hunting, compare raw, bandpassed, AGC, time-gained, and f-k-filtered gathers cautiously; f-k filtering can suppress useful diffraction energy.
